# Publication Figure: Inhibition profiles & Nutrient Factors

This notebook builds a **synthetic, publication-focused comparison** of inhibition profiles implemented in `animakinetica/models/kinetics.py`.

## Scope and assumptions
- Uses inhibition equations exactly as implemented in `KineticModel.calculate_inhibition_factors` and `KineticModel.inhibition_factor`.
- Figure is **not** coupled to an ODE simulation (no `CellCultureSimulation` run).
- Main panel uses **normalized concentration axes** for cross-inhibitor comparison.
- Supplemental panels show **absolute concentration units** for interpretability.
- Uses current default profile choices from the model setup:
  - Lactate: `power_decay`
  - Ammonia: `power_decay`
  - Osmolality: `linear`
  - Oxygen and CO₂: dedicated model equations in `kinetics.py`.


In [1]:
import sys
from pathlib import Path
import importlib

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Ensure package import from workspace
workspace_root = Path.cwd().parent
src_path = workspace_root / "animakinetica" / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Reload modules during iterative development
import animakinetica.core.parameters as parameters_module
import animakinetica.models.kinetics as kinetics_module
import animakinetica.models.stoichiometry as stoichiometry_module

importlib.reload(parameters_module)
importlib.reload(kinetics_module)
importlib.reload(stoichiometry_module)

SimulationParameters = parameters_module.SimulationParameters
Kinetics = kinetics_module.Kinetics
create_kinetic_model = kinetics_module.create_kinetic_model
StoichiometricModel = stoichiometry_module.StoichiometricModel
StoichiometryMode = stoichiometry_module.StoichiometryMode


# -----------------------------------------------------------------------------
# 1) Load reference parameters and build kinetics object (no ODE simulation)
# -----------------------------------------------------------------------------
params = SimulationParameters.load_parameters([
    "base_parameters",
    "cell_properties/cell_GFI",
    "stoichiometry/anab_GFI",
    "stoichiometry/cata_GFI",
])

stoich_model = StoichiometricModel(params=params, mode=StoichiometryMode.CONSTANT)
km = create_kinetic_model(Kinetics.STOICHIOMETRIC, params, stoich_model)

# Active inhibition model selections
model_lac = km.inhibition_model_Lac
model_amm = km.inhibition_model_Amm
model_osm = km.inhibition_model_Osm

# Parameter aliases used by inhibition equations
IC50_Lac = km.IC50_Lac
IC100_Lac = km.IC100_Lac
phi_Lac = km.phi_power_decay_Lac

IC50_Amm = km.IC50_Amm
IC100_Amm = km.IC100_Amm
phi_Amm = km.phi_power_decay_Amm

C_O2_inh_min = km.C_O2_inh_min
IC50_O2 = km.IC50_O2
n_O2 = km.n_O2

IC100_CO2 = km.IC100_CO2
n_CO2 = km.n_CO2

Osm_baseline = km.Osm_baseline_inhibition
IC50_Osm_eff = km.IC50_Osm_effective
IC100_Osm_eff = km.IC100_Osm_effective
phi_Osm = getattr(km, "phi_power_decay_Osm", None)

print("Inhibition profile setup loaded")
print(f"Models: Lac={model_lac}, Amm={model_amm}, Osm={model_osm}")


# -----------------------------------------------------------------------------
# 2) Synthetic concentration grid (normalized)
# -----------------------------------------------------------------------------
x_norm = np.linspace(0.0, 1.0, 600)

def _conc_from_norm(x: np.ndarray, model: str, ic50: float | None, ic100: float | None) -> np.ndarray:
    if model in ("power_decay", "linear"):
        return x * float(ic100)
    if model in ("exponential", "hyperbolic"):
        return x * float(ic50)
    raise ValueError(f"Unknown inhibition model: {model}")

# Normalized-to-absolute mappings used for evaluation
C_Lac_norm = _conc_from_norm(x_norm, model_lac, IC50_Lac, IC100_Lac)
C_Amm_norm = _conc_from_norm(x_norm, model_amm, IC50_Amm, IC100_Amm)
C_O2_norm = C_O2_inh_min + x_norm * IC50_O2
C_CO2_norm = x_norm * IC100_CO2

# Osmolality uses absolute deviation from baseline
if model_osm in ("power_decay", "linear"):
    Osm_eff_norm = x_norm * IC100_Osm_eff
else:
    Osm_eff_norm = x_norm * IC50_Osm_eff
Osm_norm = Osm_baseline + Osm_eff_norm


# -----------------------------------------------------------------------------
# 3) Evaluate inhibition factors using kinetics.py equations
# -----------------------------------------------------------------------------
def IF_lac(C: np.ndarray) -> np.ndarray:
    return km.inhibition_factor(
        C,
        model_lac,
        IC50=IC50_Lac,
        IC100=IC100_Lac,
        phi_power_decay=phi_Lac,
    )

def IF_amm(C: np.ndarray) -> np.ndarray:
    return km.inhibition_factor(
        C,
        model_amm,
        IC50=IC50_Amm,
        IC100=IC100_Amm,
        phi_power_decay=phi_Amm,
    )

def IF_o2(C: np.ndarray) -> np.ndarray:
    C_safe = np.maximum(C, 0.0)
    C_inhibiting = np.maximum(C_safe - C_O2_inh_min, 0.0)
    IF = (IC50_O2 ** n_O2) / ((IC50_O2 ** n_O2) + np.power(C_inhibiting, n_O2))
    return np.clip(IF, 0.0, 1.0)

def IF_co2(C: np.ndarray) -> np.ndarray:
    C_safe = np.maximum(C, 0.0)
    IF = 1.0 - np.power(C_safe, n_CO2) / (IC100_CO2 ** n_CO2)
    return np.clip(IF, 0.0, 1.0)

def IF_osm(Osm: np.ndarray) -> np.ndarray:
    Osm_eff = np.abs(Osm - Osm_baseline)
    return km.inhibition_factor(
        Osm_eff,
        model_osm,
        IC50=IC50_Osm_eff,
        IC100=IC100_Osm_eff,
        phi_power_decay=phi_Osm,
    )

IF_Lac_norm = IF_lac(C_Lac_norm)
IF_Amm_norm = IF_amm(C_Amm_norm)
IF_O2_norm = IF_o2(C_O2_norm)
IF_CO2_norm = IF_co2(C_CO2_norm)
IF_Osm_norm = IF_osm(Osm_norm)


# -----------------------------------------------------------------------------
# 4) Supplementary IC table (absolute values)
# -----------------------------------------------------------------------------
ic_table_rows = [
    {
        "Compound": "Lactate",
        "IC50": IC50_Lac,
        "IC100": IC100_Lac,
        "Unit": "mmol/L",
    },
    {
        "Compound": "Ammonia",
        "IC50": IC50_Amm,
        "IC100": IC100_Amm,
        "Unit": "mmol/L",
    },
    {
        "Compound": "Oxygen",
        "IC50": IC50_O2,
        "IC100": np.nan,
        "Unit": "mmol/L",
    },
    {
        "Compound": "CO2",
        "IC50": np.nan,
        "IC100": IC100_CO2,
        "Unit": "mmol/L",
    },
    {
        "Compound": "Osmolality (effective)",
        "IC50": IC50_Osm_eff,
        "IC100": IC100_Osm_eff,
        "Unit": "mOsm",
    },
]

ic_table_df = pd.DataFrame(ic_table_rows)
display(ic_table_df.style.format({"IC50": "{:.3g}", "IC100": "{:.3g}"}))


# -----------------------------------------------------------------------------
# 5) Publication figure: normalized panel with dedicated legend panel
# -----------------------------------------------------------------------------
inh_color_map = {
    "Lactate": "#1f77b4",
    "Ammonia": "#ff7f0e",
    "Oxygen": "#2ca02c",
    "CO2": "#d62728",
    "Osmolality": "#9467bd",
}

inh_dash_map = {
    "Lactate": "solid",
    "Ammonia": "dash",
    "Oxygen": "dot",
    "CO2": "dashdot",
    "Osmolality": "longdash",
}

line_width_inh = 8

fig = make_subplots(
    rows=1,
    cols=1,
    horizontal_spacing=0.10,
    vertical_spacing=0.12,
)

trace_defs = [
    ("Lactate", x_norm, IF_Lac_norm),
    ("Ammonia", x_norm, IF_Amm_norm),
    ("Oxygen", x_norm, IF_O2_norm),
    ("CO2", x_norm, IF_CO2_norm),
    ("Osmolality", x_norm, IF_Osm_norm),
]

for name, x_n, y_n in trace_defs:
    fig.add_trace(
        go.Scatter(
            x=x_n,
            y=y_n,
            mode="lines",
            name=(
                "Lactate (C<sub>Lac</sub>)" if name == "Lactate" else
                "Ammonia (C<sub>Amm</sub>)" if name == "Ammonia" else
                "Oxygen (C<sub>O2</sub>)" if name == "Oxygen" else
                "CO<sub>2</sub> (C<sub>CO2</sub>)" if name == "CO2" else
                "Osmolality"
            ),
            line=dict(width=line_width_inh, color=inh_color_map[name], dash=inh_dash_map[name]),
            showlegend=True,
        ),
        row=1,
        col=1,
    )

def _fmt_legend(v):
    return f"{float(v):.3g}"

lac_ic_name, lac_ic_value = (
    ("IC100", IC100_Lac) if model_lac in ("power_decay", "linear") else ("IC50", IC50_Lac)
)
amm_ic_name, amm_ic_value = (
    ("IC100", IC100_Amm) if model_amm in ("power_decay", "linear") else ("IC50", IC50_Amm)
)
osm_ic_name, osm_ic_value = (
    ("IC100", IC100_Osm_eff) if model_osm in ("power_decay", "linear") else ("IC50", IC50_Osm_eff)
)

inh_labels = {
    "Lactate": "Lactate", #f"Lactate (C<sub>Lac</sub>; {lac_ic_name}={_fmt_legend(lac_ic_value)} mmol/L)",
    "Ammonia": "Ammonia", # f"Ammonia (C<sub>Amm</sub>; {amm_ic_name}={_fmt_legend(amm_ic_value)} mmol/L)",
    "Oxygen": "O<sub>2</sub>", 
    # (
    #     f"Oxygen (C<sub>O2</sub>; C<sub>O2,min</sub>={_fmt_legend(C_O2_inh_min)} mmol/L; "
    #     f"IC50={_fmt_legend(IC50_O2)} mmol/L)"
    # ),
    "CO2": f"CO<sub>2</sub>", # f"CO<sub>2</sub>",# (C<sub>CO2</sub>; IC100={_fmt_legend(IC100_CO2)} mmol/L)",
    "Osmolality": "Osmolality", 
    # (
    #     f"Osmolality (baseline={_fmt_legend(Osm_baseline)} mOsm; "
    #     f"{osm_ic_name}={_fmt_legend(osm_ic_value)} mOsm)"
    # ),
}


def _inh_class(name: str) -> str | None:
    if name.startswith("Lactate"):
        return "Lactate"
    if name.startswith("Ammonia"):
        return "Ammonia"
    if name.startswith("Oxygen"):
        return "Oxygen"
    if name.startswith("CO<sub>2</sub>"):
        return "CO2"
    if name.startswith("Osmolality"):
        return "Osmolality"
    return None

# Update colored figure legend labels and re-show
for trace in fig.data:
    cls = _inh_class(trace.name)
    if cls is not None:
        trace.name = inh_labels[cls]


# Guide lines for normalized panel
fig.add_hline(y=1.0, line_dash="dot", line_color="black", opacity=0.35, row=1, col=1)
fig.add_hline(y=0.5, line_dash="dash", line_color="black", opacity=0.25, row=1, col=1)
fig.add_hline(y=0.0, line_dash="dot", line_color="black", opacity=0.2, row=1, col=1)
fig.add_vline(x=1.0, line_dash="dash", line_color="gray", opacity=0.35, row=1, col=1)

# Axis labels
fig.update_xaxes(
    title_text="C / C<sub>inh</sub>",
    row=1,
    col=1,
    range=[0.0, 1.0],
    showline=True,
    linewidth=3,
    linecolor="black",
    ticks="outside",
    tickwidth=2,
)
fig.update_yaxes(
    title_text="μ/μ<sub>max</sub>",
    row=1,
    col=1,
    range=[0.0, 1.05],
    showline=True,
    linewidth=3,
    linecolor="black",
    ticks="outside",
    tickwidth=2,
)

# Hide unused panels
fig.update_xaxes(visible=False, showgrid=False, zeroline=False, row=1, col=2)
fig.update_yaxes(visible=False, showgrid=False, zeroline=False, row=1, col=2)
fig.update_xaxes(visible=False, showgrid=False, zeroline=False, row=2, col=1)
fig.update_yaxes(visible=False, showgrid=False, zeroline=False, row=2, col=1)
fig.update_xaxes(visible=False, showgrid=False, zeroline=False, row=2, col=2)
fig.update_yaxes(visible=False, showgrid=False, zeroline=False, row=2, col=2)

fig.update_layout(
    template="simple_white",
    width=700,
    height=700,
    font=dict(size=30),
    legend=dict(
        orientation="v",
        yanchor="bottom",
        y=0.02,
        xanchor="left",
        x=0.02,
        bgcolor="rgba(0,0,0,0)",
        bordercolor="rgba(0,0,0,0)",
        borderwidth=0,
        font=dict(size = 30),

    ),
    margin=dict(l=120, r=0, t=0, b=120),
)

fig.show()

# Keep notebook view, but make exported figure transparent
fig.update_layout(
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    legend=dict(bgcolor="rgba(0,0,0,0)"),
)


# -----------------------------------------------------------------------------
# 6) Optional export for manuscript workflow
# -----------------------------------------------------------------------------
SAVE_EXPORTS = True

if SAVE_EXPORTS:
    out_dir = Path.cwd()
    svg_path = out_dir / "inhibition_profiles.svg"
    # csv_path = out_dir / "inhibition_profile_ic_table.csv"

    # ic_table_df.to_csv(csv_path, index=False)
    # print(f"Saved IC table: {csv_path}")

    try:
        fig.write_image(str(svg_path))
        print(f"Saved figure: {svg_path}")
    except Exception as exc:
        print("Image export skipped (install/enable kaleido if needed).")
        print(f"Export error: {exc}")


Loading parameters from 4 file(s) sequentially...
  [1/4] Base: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/4] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\cell_properties\cell_GFI.yaml
  [3/4] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\stoichiometry\anab_GFI.yaml
  [4/4] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\stoichiometry\cata_GFI.yaml
Successfully loaded parameters from 4 file(s).
Inhibition profile setup loaded
Models: Lac=power_decay, Amm=power_decay, Osm=linear


,Compound,IC50,IC100,Unit
0,Lactate,39.9,72.9,mmol/L
1,Ammonia,4.08,8.15,mmol/L
2,Oxygen,0.22,nan,mmol/L
3,CO2,nan,8.6,mmol/L
4,Osmolality (effective),165,330,mOsm


Saved figure: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Method Figures\inhibition_profiles.svg


In [2]:
# Additional rendering in grayscale (same plot, colors only)

# B/W post-processing from the updated colored figure
inhibition_gray_map = {
    "Lactate": "#000000",
    "Ammonia": "#3a3a3a",
    "Oxygen": "#666666",
    "CO2": "#8c8c8c",
    "Osmolality": "#b0b0b0",
}


def _inh_class_bw(name: str) -> str | None:
    n = (name or "").strip().lower()
    if n.startswith("lactate"):
        return "Lactate"
    if n.startswith("ammonia"):
        return "Ammonia"
    if n.startswith("oxygen") or n.startswith("o<sub>2</sub>") or n.startswith("o2"):
        return "Oxygen"
    if n.startswith("co<sub>2</sub>") or n.startswith("co2"):
        return "CO2"
    if n.startswith("osmolality"):
        return "Osmolality"
    return None


fig_bw = go.Figure(fig)

for trace in fig_bw.data:
    cls = _inh_class_bw(trace.name)
    if cls in inhibition_gray_map:
        trace.line.color = inhibition_gray_map[cls]

# Slightly thicker axis lines
fig_bw.update_xaxes(showline=True, linewidth=3, linecolor="black", ticks="outside", tickwidth=2)
fig_bw.update_yaxes(showline=True, linewidth=3, linecolor="black", ticks="outside", tickwidth=2)

# Keep identical guide-line structure, recolor only
if fig_bw.layout.shapes:
    for i, shp in enumerate(fig_bw.layout.shapes):
        shp.line.color = "#000000" if i <= 7 else "#5a5a5a"

fig_bw.show()


### Figure caption (publishable)

**Figure X. Inhibition and nutrient factor profiles used in the kinetic model.**
**(A) Inhibitory factor profiles** ($\mu/\mu_{max}$) as functions of normalized inhibitor concentration. Curves were generated from the analytical inhibition formulations implemented in `KineticModel.inhibition_factor` and `KineticModel.calculate_inhibition_factors` (`animakinetica/models/kinetics.py`) using the active model selections shown in the legend. Lactate and ammonia are parameterized with the model-specific inhibitory constants (here `power_decay`, normalized to $IC_{100}$), oxygen is represented by a thresholded inhibition term normalized with $(C_{O2}-C_{O2,min})/IC_{50,O2}$, carbon dioxide is normalized to $IC_{100,CO2}$, and osmolality is expressed as effective deviation from baseline, $|Osm-Osm_{baseline}|$, including the baseline value in the legend. 

**(B) Nutrient factor profiles** ($\mu/\mu_{max}$) as Monod-type nutrient limitation terms for glucose, glutamine, and oxygen, shown against normalized concentration $C/C_{ref}$. Legends report both the corresponding Monod constant ($K_{Glc}$, $K_{Gln}$, $K_{O2}$) and the reference concentration ($C_{ref}$), with units. 

All curves are synthetic parameter sweeps (not dynamic cultivation trajectories) and are intended to isolate and visualize the standalone response of each kinetic factor.

# Nutrient factors

In [3]:
# Nutrient factor profiles
# x-axis: C/C_ref, y-axis: mu/mu_max contribution from each Monod term


def _to_float(x):
    return float(x.value) if hasattr(x, "value") else float(x)


def _fmt_legend(v):
    return f"{float(v):.3g}"


# Monod constants from active kinetics object
K_Glc = _to_float(getattr(km, "K_Glc", params.kinetics.K_Glc))
K_Gln = _to_float(getattr(km, "K_Gln", params.kinetics.K_Gln))
K_O2 = _to_float(getattr(km, "K_O2", params.kinetics.K_O2))

# Requested C_ref values
Cmax_Glc = 5.0   # mmol/L
Cmax_Gln = 5.0   # mmol/L
Cmax_O2 = 0.01   # mmol/L

x_norm_nf = np.linspace(0.0, 1.0, 600)
C_Glc_abs = x_norm_nf * Cmax_Glc
C_Gln_abs = x_norm_nf * Cmax_Gln
C_O2_abs_nf = x_norm_nf * Cmax_O2

# Monod nutrient factors (equal to mu/mu_max contribution when other factors are 1)
NF_Glc = C_Glc_abs / (K_Glc + C_Glc_abs)
NF_Gln = C_Gln_abs / (K_Gln + C_Gln_abs)
NF_O2 = C_O2_abs_nf / (K_O2 + C_O2_abs_nf)

nutrient_labels = {
    "Glucose": "Glucose",
    # (
    #     f"Glucose (C<sub>Glc</sub>; K<sub>Glc</sub>={_fmt_legend(K_Glc)} mmol/L; "
    #     f"C<sub>ref</sub>={_fmt_legend(Cmax_Glc)} mmol/L)"
    # ),
    "Glutamine": "Glutamine",
    # (
    #     f"Glutamine (C<sub>Gln</sub>; K<sub>Gln</sub>={_fmt_legend(K_Gln)} mmol/L; "
    #     f"C<sub>ref</sub>={_fmt_legend(Cmax_Gln)} mmol/L)"
    # ),
    "Oxygen": "O<sub>2</sub>",
    # (
    #     f"Oxygen (C<sub>O2</sub>; K<sub>O2</sub>={_fmt_legend(K_O2)} mmol/L; "
    #     f"C<sub>ref</sub>={_fmt_legend(Cmax_O2)} mmol/L)"
    # ),
}

# Colored base figure
nutrient_color_map = {
    nutrient_labels["Glucose"]: "#1f77b4",
    nutrient_labels["Glutamine"]: "#ff7f0e",
    nutrient_labels["Oxygen"]: "#2ca02c",
}

nutrient_dash_map = {
    nutrient_labels["Glucose"]: "solid",
    nutrient_labels["Glutamine"]: "dash",
    nutrient_labels["Oxygen"]: "dot",
}

fig_nf = go.Figure()

for name, y in [
    (nutrient_labels["Glucose"], NF_Glc),
    (nutrient_labels["Glutamine"], NF_Gln),
    (nutrient_labels["Oxygen"], NF_O2),
]:
    fig_nf.add_trace(
        go.Scatter(
            x=x_norm_nf,
            y=y,
            mode="lines",
            name=name,
            line=dict(color=nutrient_color_map[name], width=8, dash=nutrient_dash_map[name]),
        )
    )

fig_nf.add_hline(y=1.0, line_dash="dot", line_color="black", opacity=0.35)
fig_nf.add_hline(y=0.5, line_dash="dash", line_color="black", opacity=0.25)
fig_nf.add_hline(y=0.0, line_dash="dot", line_color="black", opacity=0.2)
fig_nf.add_vline(x=1.0, line_dash="dash", line_color="gray", opacity=0.35)

fig_nf.update_xaxes(
    title_text="C / C<sub>ref</sub>",
    range=[0.0, 1.0],
    showline=True,
    linewidth=3,
    linecolor="black",
    ticks="outside",
    tickwidth=2,
)
fig_nf.update_yaxes(
    title_text="μ/μ<sub>max</sub>",
    range=[0.0, 1.05],
    showline=True,
    linewidth=3,
    linecolor="black",
    ticks="outside",
    tickwidth=2,
)

fig_nf.update_layout(
    template="simple_white",
    width=700,
    height=700,
    font=dict(size=30),
    legend=dict(
        orientation="v",
        yanchor="bottom",
        y=0.02,
        xanchor="right",
        x=0.98,
        bgcolor="rgba(255,255,255,0.0)",
        bordercolor="rgba(0,0,0,0)",
        borderwidth=0,
        font=dict(size=30),
    ),
    margin=dict(l=120, r=0, t=0, b=120),
)

fig_nf.show()


# Keep notebook view, but make exported figure transparent
fig_nf.update_layout(
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    legend=dict(bgcolor="rgba(0,0,0,0)"),
)


# -----------------------------------------------------------------------------
# 6) Optional export for manuscript workflow
# -----------------------------------------------------------------------------
SAVE_EXPORTS = True

if SAVE_EXPORTS:
    out_dir = Path.cwd()
    svg_path = out_dir / "nutrient_profiles.svg"
    # csv_path = out_dir / "inhibition_profile_ic_table.csv"

    # ic_table_df.to_csv(csv_path, index=False)
    # print(f"Saved IC table: {csv_path}")

    try:
        fig_nf.write_image(str(svg_path))
        print(f"Saved figure: {svg_path}")
    except Exception as exc:
        print("Image export skipped (install/enable kaleido if needed).")
        print(f"Export error: {exc}")




Saved figure: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Method Figures\nutrient_profiles.svg


In [4]:

# Additional rendering in grayscale (post-processing of colored figure)
nutrient_gray_map = {
    "Glucose": "#000000",
    "Glutamine": "#3a3a3a",
    "Oxygen": "#666666",
}


def _nf_class(name: str) -> str | None:
    if name.startswith("Glucose"):
        return "Glucose"
    if name.startswith("Glutamine"):
        return "Glutamine"
    if name.startswith("Oxygen") or name.startswith("O<sub>2</sub>") or name.startswith("O2"):
        return "Oxygen"
    return None


fig_nf_bw = go.Figure(fig_nf)

for trace in fig_nf_bw.data:
    cls = _nf_class(trace.name)
    if cls in nutrient_gray_map:
        trace.line.color = nutrient_gray_map[cls]

# Slightly thicker axis lines
fig_nf_bw.update_xaxes(showline=True, linewidth=3, linecolor="black", ticks="outside", tickwidth=2)
fig_nf_bw.update_yaxes(showline=True, linewidth=3, linecolor="black", ticks="outside", tickwidth=2)

if fig_nf_bw.layout.shapes:
    for i, shp in enumerate(fig_nf_bw.layout.shapes):
        shp.line.color = "#000000" if i <= 2 else "#5a5a5a"

fig_nf_bw.show()



In [5]:

# Supplementary table: Monod constants and C_ref
supp_nf_table_df = pd.DataFrame([
    {
        "Nutrient": "Glucose",
        "K_Monod": K_Glc,
        "C_ref": Cmax_Glc,
        "Unit": "mmol/L",
    },
    {
        "Nutrient": "Glutamine",
        "K_Monod": K_Gln,
        "C_ref": Cmax_Gln,
        "Unit": "mmol/L",
    },
    {
        "Nutrient": "Oxygen",
        "K_Monod": K_O2,
        "C_ref": Cmax_O2,
        "Unit": "mmol/L",
    },
])

supp_nf_table_df["K_Monod/C_ref"] = supp_nf_table_df["K_Monod"] / supp_nf_table_df["C_ref"]

display(
    supp_nf_table_df.style.format({
        "K_Monod": "{:.4g}",
        "C_ref": "{:.4g}",
        "K_Monod/C_ref": "{:.4g}",
    })
)


,Nutrient,K_Monod,C_ref,Unit,K_Monod/C_ref
0,Glucose,0.44,5,mmol/L,0.088
1,Glutamine,0.28,5,mmol/L,0.056
2,Oxygen,8e-06,0.01,mmol/L,0.0008
